# GRIB: indexing metadata with a database

Scanning many GRIB files to find the fields we need can be slow, because every message has to be read just to get its metadata. With the `index` option of `to_fieldlist()` earthkit-data scans the GRIB files **once** and stores the metadata of each message in a small database (sqlite). Next time the fieldlist is built from the database, without reading the GRIB files again. The GRIB data are only read when the values are really needed.

In this notebook we show how to use it.

### Indexing with `index=True`

With `index=True` every GRIB file gets its own database stored in the earthkit-data cache. Because of this the cache must be enabled, so we set the `cache-policy` to `"temporary"` (the cache is removed at the end of the session). With the default policy `"off"` an error is raised.

In [1]:
import earthkit.data as ekd

ekd.config.set("cache-policy", "temporary")

ds = ekd.from_source("sample", "tuv_pl.grib")
fl = ds.to_fieldlist(index=True)
len(fl)

tuv_pl.grib:   0%|          | 0.00/4.22k [00:00<?, ?B/s]

0it [00:00, ?it/s]

1it [00:00,  1.67it/s]

18

The result is a normal fieldlist. We can use it as usual:

In [2]:
fl.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
1,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
2,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
3,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
4,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
5,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
6,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,700,pressure,0,regular_ll
7,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,700,pressure,0,regular_ll
8,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,700,pressure,0,regular_ll
9,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,500,pressure,0,regular_ll


The database was created in the cache. We can check it with the (low-level) `GribIndex` class:

In [3]:
from earthkit.data.readers.grib.index import GribIndex

# find_in_cache() returns None when there is no up-to-date index; otherwise
# it returns an open database that we must close when we are done
with GribIndex.find_in_cache(ds.path) as index:
    print(f"{index.count()} messages are indexed")

18 messages are indexed


### Indexing several files into one database

We can also store the index of many files in a single database by passing the path of the database as `index`. It is only created when it does not exist yet.

First we prepare three small GRIB files, one for each parameter, in a temporary directory.

In [4]:
import os
import tempfile

tmp = tempfile.TemporaryDirectory()

paths = []
for name in ["t", "u", "v"]:
    path = os.path.join(tmp.name, f"{name}_pl.grib")
    fl.sel({"parameter.variable": name}).to_target("file", path)
    paths.append(path)

[os.path.basename(p) for p in paths]

['t_pl.grib', 'u_pl.grib', 'v_pl.grib']

Now we build a fieldlist from the three files, storing the index in `index.sqlite`:

In [5]:
db_path = os.path.join(tmp.name, "index.sqlite")

ds = ekd.from_source("file", paths)
fl = ds.to_fieldlist(index=db_path)
fl.ls()

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
1,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
2,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,700,pressure,0,regular_ll
3,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,500,pressure,0,regular_ll
4,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,400,pressure,0,regular_ll
5,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,300,pressure,0,regular_ll
6,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
7,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
8,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,700,pressure,0,regular_ll
9,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,500,pressure,0,regular_ll


We can look into the database. It knows the GRIB files and the number of messages in them:

In [6]:
with GribIndex.open(paths, db_path=db_path) as index:
    for p in index.paths():
        print(os.path.basename(p), index.count(p))
    print("total:", index.count())

t_pl.grib 6
u_pl.grib 6
v_pl.grib 6
total: 18


### Reusing the database

When the same database is used again nothing is scanned: it is reused as long as the GRIB files have not changed (checked by their size and modification time). A file that was modified is scanned again, and new files are added to the database. We can see that the database is untouched by the second call:

In [7]:
before = os.stat(db_path)

fl2 = ekd.from_source("file", paths).to_fieldlist(index=db_path)

after = os.stat(db_path)
print(len(fl2), before.st_size == after.st_size, before.st_mtime_ns == after.st_mtime_ns)

18 True True


### Storing extra keys

By default the database stores the metadata keys of the fieldlist components (e.g. `parameter.variable`, `time.step`, `vertical.level`) and the most commonly used GRIB keys (e.g. `shortName`, `level`, `date`). Any other GRIB key has to be read from the GRIB file. If we need other GRIB keys often, we can store them with `index_extra_keys`. They are available as `metadata.<key>`.

In [8]:
db_path_extra = os.path.join(tmp.name, "index_extra.sqlite")

fl3 = ds.to_fieldlist(index=db_path_extra, index_extra_keys=["bitsPerValue"])
fl3.get("metadata.bitsPerValue")[:6]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

[4, 4, 4, 4, 4, 4]

### Selecting and getting metadata

The metadata of an indexed fieldlist comes from the database, so selecting fields and getting metadata values does not need to read the GRIB files.

In [9]:
fl_sel = fl.sel({"parameter.variable": ["t", "u"], "vertical.level": [500, 850]})
fl_sel.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
1,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,500,pressure,0,regular_ll
2,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
3,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,500,pressure,0,regular_ll


In [10]:
fl_sel.get(["parameter.variable", "vertical.level", "metadata.shortName"])

[['t', 850, 't'], ['t', 500, 't'], ['u', 850, 'u'], ['u', 500, 'u']]

The values are read from the GRIB file only when we ask for them:

In [11]:
fl_sel[0].to_numpy().shape

(7, 12)

### Converting to Xarray

An indexed fieldlist can be converted into an Xarray dataset the usual way. The GRIB data are only read when the values are loaded.

In [12]:
ds_xr = fl.to_xarray()
ds_xr

<xarray.Dataset> Size: 12kB
Dimensions:    (level: 6, latitude: 7, longitude: 12)
Coordinates:
  * level      (level) int64 48B 300 400 500 700 850 1000
  * latitude   (latitude) float64 56B 90.0 60.0 30.0 0.0 -30.0 -60.0 -90.0
  * longitude  (longitude) float64 96B 0.0 30.0 60.0 90.0 ... 270.0 300.0 330.0
Data variables:
    t          (level, latitude, longitude) float64 4kB ...
    u          (level, latitude, longitude) float64 4kB ...
    v          (level, latitude, longitude) float64 4kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

In [13]:
ds_xr["t"].sel(level=500).load().values.shape

(7, 12)

In [14]:
# remove the temporary files
tmp.cleanup()